In [1]:
# 01_download_data.py

import yfinance as yf
import pandas as pd

tickers = [
    "^GSPC",      # S&P500
    "BTC-USD",    # Bitcoin
    "GC=F"        # Gold
]

for ticker in tickers:

    print(f"Downloading {ticker}")

    df = yf.download(
        ticker,
        start="2015-01-01",
        end="2025-01-01",
        auto_adjust=True
    )

    print(df.head())

    filename = ticker.replace("^", "")

    df.to_csv(f"data/findata.csv")

print("Download Complete")

[*********************100%***********************]  1 of 1 completed


Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2015-01-02  2058.199951  2072.360107  2046.040039  2058.899902  2708700000
2015-01-05  2020.579956  2054.439941  2017.339966  2054.439941  3799120000
2015-01-06  2002.609985  2030.250000  1992.439941  2022.150024  4460110000
2015-01-07  2025.900024  2029.609985  2005.550049  2005.550049  3805480000
2015-01-08  2062.139893  2064.080078  2030.609985  2030.609985  3934010000


[*********************100%***********************]  1 of 1 completed


Price            Close        High         Low        Open    Volume
Ticker         BTC-USD     BTC-USD     BTC-USD     BTC-USD   BTC-USD
Date                                                                
2015-01-01  314.248993  320.434998  314.002991  320.434998   8036550
2015-01-02  315.032013  315.838989  313.565002  314.079010   7860650
2015-01-03  281.082001  315.149994  281.082001  314.846008  33054400
2015-01-04  264.195007  287.230011  257.612000  281.145996  55629100
2015-01-05  274.473999  278.341003  265.084015  265.084015  43962800


[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open Volume
Ticker             GC=F         GC=F         GC=F         GC=F   GC=F
Date                                                                 
2015-01-02  1186.000000  1194.500000  1169.500000  1184.000000    138
2015-01-05  1203.900024  1206.900024  1180.099976  1180.300049    470
2015-01-06  1219.300049  1220.000000  1203.500000  1203.500000     97
2015-01-07  1210.599976  1219.199951  1210.599976  1219.199951     29
2015-01-08  1208.400024  1215.699951  1206.300049  1207.000000     92
Download Complete


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/findata.csv")

df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
prices = df["Close"]

returns = np.log(prices / prices.shift(1))
returns = returns.dropna()

print(returns.head())

returns.to_csv(
    "data/findata_returns.csv",
    index=False
)

3    0.014980
4    0.012711
5   -0.007161
6   -0.001819
7    0.006270
Name: Close, dtype: float64


In [3]:
# 03_garch_model.py

import pandas as pd

from arch import arch_model

returns = pd.read_csv(
    "data/findata_returns.csv"
)

returns = returns.squeeze()

model = arch_model(
    returns * 100,
    p=1,
    q=1
)

results = model.fit()

print(results.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 14631.70379847351
Iteration:      2,   Func. Count:     15,   Neg. LLF: 711733833536.7637
Iteration:      3,   Func. Count:     24,   Neg. LLF: 3874.481261178248
Iteration:      4,   Func. Count:     32,   Neg. LLF: 63020683.302703254
Iteration:      5,   Func. Count:     38,   Neg. LLF: 56348.37966910278
Iteration:      6,   Func. Count:     45,   Neg. LLF: 3928.418023712503
Iteration:      7,   Func. Count:     51,   Neg. LLF: 3295.52364453446
Iteration:      8,   Func. Count:     57,   Neg. LLF: 3289.1274103369587
Iteration:      9,   Func. Count:     63,   Neg. LLF: 3280.946364398199
Iteration:     10,   Func. Count:     69,   Neg. LLF: 3280.146501123302
Iteration:     11,   Func. Count:     75,   Neg. LLF: 3280.0916160821253
Iteration:     12,   Func. Count:     81,   Neg. LLF: 3280.0580988471747
Iteration:     13,   Func. Count:     87,   Neg. LLF: 3280.0578690798466
Iteration:     14,   Func. Count:     92,   Neg. LLF: 3280.0

In [4]:
volatility = results.conditional_volatility

print(volatility.head())

volatility.to_csv(
    "data/volatility.csv",
    index=False
)

0    1.077722
1    1.088807
2    1.091696
3    1.081707
4    1.065671
Name: cond_vol, dtype: float64


In [5]:
# 04_transformer_model.py

import pandas as pd
import numpy as np

volatility = pd.read_csv(
    "data/volatility.csv"
)

volatility = volatility.values

print(volatility.shape)

(2512, 1)


In [6]:
sequence_length = 30

X = []
y = []

for i in range(
    len(volatility) - sequence_length
):

    X.append(
        volatility[i:i+sequence_length]
    )

    y.append(
        volatility[i+sequence_length]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(2482, 30, 1)
(2482, 1)


In [7]:
import torch
import torch.nn as nn

class VolatilityTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Linear(
            1,
            64
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.fc = nn.Linear(
            64,
            1
        )

    def forward(self,x):

        x = self.embedding(x)

        x = self.transformer(x)

        x = x[:,-1,:]

        return self.fc(x)

model = VolatilityTransformer()

print(model)

VolatilityTransformer(
  (embedding): Linear(in_features=1, out_features=64, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [8]:
# 05_train.py

import torch

print(
    "Torch Version:",
    torch.__version__
)

print(
    "CUDA:",
    torch.cuda.is_available()
)

Torch Version: 2.12.0+cpu
CUDA: False


In [9]:
sample = torch.randn(
    32,
    30,
    1
)

output = model(sample)

print(output.shape)

torch.Size([32, 1])
